# Multi-disease human liver validation

This notebook inspects frozen patient-, biopsy-, and section-level results from MASLD, primary sclerosing cholangitis (PSC), and alcohol-associated hepatitis (AH). Patient remains the biological unit; the four PSC sections derive from one patient.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if not (root / 'examples').is_dir():
    root = root.parent
evidence = root / 'examples' / 'manuscript_evidence' / 'figure_06'
masld = pd.read_csv(evidence / '04_PATIENT_LEVEL_RESULTS_MASLD.tsv', sep='\t')
psc = pd.read_csv(evidence / '04A_SECTION_LEVEL_RESULTS_PSC.tsv', sep='\t')
ah = pd.read_csv(evidence / '04_PATIENT_LEVEL_RESULTS.tsv', sep='\t')

In [ ]:
columns = ['disease', 'patient', 'median_spatial_idw', 'median_spatial_map', 'log1p_rmse_idw', 'log1p_rmse_map', 'pooled_residual_pearson_map']
units = pd.concat([masld[columns], psc[columns], ah[columns]], ignore_index=True)
summary = units.groupby('disease').agg(evaluation_units=('patient','size'), spatial_IDW=('median_spatial_idw','median'), spatial_MAP=('median_spatial_map','median'), RMSE_IDW=('log1p_rmse_idw','median'), RMSE_MAP=('log1p_rmse_map','median'), residual_r=('pooled_residual_pearson_map','median'))
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
colors = {'MASLD':'#3B7EA1', 'PSC':'#D17C35', 'AH':'#7A5195'}
for disease, group in units.groupby('disease', sort=False):
    color = colors[disease]
    for _, row in group.iterrows():
        axes[0].plot([0, 1], [row.median_spatial_idw, row.median_spatial_map], color=color, alpha=0.25, lw=0.7)
        axes[1].plot([0, 1], [row.log1p_rmse_idw, row.log1p_rmse_map], color=color, alpha=0.25, lw=0.7)
    med = group.median(numeric_only=True)
    axes[0].plot([0, 1], [med.median_spatial_idw, med.median_spatial_map], color=color, marker='D', lw=2, label=disease)
    axes[1].plot([0, 1], [med.log1p_rmse_idw, med.log1p_rmse_map], color=color, marker='D', lw=2)
for ax, title in zip(axes, ['Spatial Pearson', 'log1p RMSE']):
    ax.set_xticks([0, 1], ['Registered IDW', 'AtlasTailor'])
    ax.set_title(title)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False)
fig.tight_layout()

MASLD contains 33 biopsy evaluations from a 32-patient cohort, PSC contains four within-patient sections from one patient, and AH contains two independent patients. These distinctions must be retained in inferential reporting.